# GPU tour: milling stability charts (branch `hill-argument-principle`)

Times full-HD milling stability charts on the GPU of this runtime, in every number format
(Float32; Float16 evaluation with a Float32 march; Float16 + a Float32 re-check of the flagged
points), each checked against a Float64 reference on a coarse grid:
* `hill/warp/tour_warp3h.jl`: the hardest case (Test 3, helix 30°/45°, 3–30 krpm × 0–10 mm) with
  the warp-cooperative kernel (one warp per point), full chart and adaptive refinement;
* `hill/gpu_tour_milling.jl`: Test 2 (`mill2g`), Test 3 with one thread per point (`mill3h`), and
  the dense Hill reference (`mill3d`, 480×270).

Pick the GPU under *Runtime → Change runtime type*, then *Run all* (~20–40 min with the Julia
setup). **Afterwards: *Runtime → Disconnect and delete runtime*.**


In [ ]:
REPO, BRANCH, JULIA_CHANNEL = "https://github.com/bachrathyd/InterpolatedNyquist.jl", "hill-argument-principle", "1.12"
REPO_DIR = '/content/InterpolatedNyquist.jl'
import os, subprocess

def sh(cmd):
    "Run a shell command, stream its output, and STOP on failure."
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'command failed (exit code {p.returncode}) -- see the output above')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
os.environ['PATH'] = '/root/.juliaup/bin:' + os.environ['PATH']
os.environ['JULIA_NUM_THREADS'] = 'auto'
if not os.path.exists('/root/.juliaup/bin/julia'):
    sh(f'curl -fsSL https://install.julialang.org | sh -s -- --yes --default-channel {JULIA_CHANNEL} > /dev/null')
if os.path.isdir(REPO_DIR):
    sh(f'cd {REPO_DIR} && git fetch -q --depth 1 origin {BRANCH} && git reset -q --hard FETCH_HEAD')
else:
    sh(f'git clone -q -b {BRANCH} --depth 1 {REPO} {REPO_DIR}')
sh(f"cd {REPO_DIR} && git log -1 --format='%h  %ad  %s' --date=short")
sh(f"cd {REPO_DIR} && julia --project=gpu/scripts -e 'using Pkg; Pkg.instantiate(); Pkg.precompile()' 2>&1 | tail -3")

In [ ]:
# hardest case, warp-cooperative kernel: smoke test (GPU vs CPU-validated reference), then full HD
# (the logs also go to /content/*.log)
sh(f"set -o pipefail; cd {REPO_DIR} && julia --project=gpu/scripts hill/warp/tour_warp3h.jl --res 320x180 "
   f"--check 48x27 --adaptive no 2>&1 | tee /content/tour_warp_smoke.log")
sh(f"set -o pipefail; cd {REPO_DIR} && julia --project=gpu/scripts hill/warp/tour_warp3h.jl --res 1920x1080 "
   f"--check 192x108 --csv /content/tour_warp.csv 2>&1 | tee /content/tour_warp.log")

In [ ]:
MODELS, ADAPTIVE = 'mill2g,mill3h,mill3d', 'yes'   # see MODELS in hill/gpu_tour_milling.jl
sh(f"set -o pipefail; cd {REPO_DIR} && julia --project=gpu/scripts hill/gpu_tour_milling.jl --models {MODELS} "
   f"--res 1920x1080 --check 192x108 --adaptive {ADAPTIVE} --csv /content/tour_main.csv 2>&1 | tee /content/tour_main.log")

In [ ]:
# registers / local memory / occupancy of the warp kernel, and the CSV rows of both tours
sh(f"set -o pipefail; cd {REPO_DIR} && julia --project=gpu/scripts hill/warp/kernel_info_warp3h.jl 2>&1 | tee /content/kinfo.log")
sh("cat /content/tour_warp.csv /content/tour_main.csv")